## 🎯 과제 2: 페르소나별 temperature 다르게 설정 (실무 빈도 ★★★★☆)

1교시에서 배웠죠. `temperature`는 페르소나의 성격과도 연결됩니다.

- **분석가**: `temperature=0` (정확하고 일관된 답변)
- **디자이너**: `temperature=0.8` (창의적이고 다양한 답변)

**요구사항**
- `PERSONAS` 사전 구조를 `{name: {"system": ..., "temperature": ...}}` 형태로 확장
- `ask()` 함수에서 페르소나별 `temperature`로 호출
- **힌트**: `ChatOpenAI(model=..., temperature=...)`을 페르소나마다 새로 만들거나, `.bind(temperature=)`을 사용

---


In [6]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ 로컬 Jupyter
✅ API 키 OK


In [10]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from datetime import datetime
print("✅ import 완료")

# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": {'system':"당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",'temperature':0},
    "마케터": {'system':"당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",'temperature':1.2},
    "개발자": {'system':"당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",'temperature':0.1},
    "기획자": {'system':"당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",'temperature':1.2},
    "디자이너": {'system':"당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.", 'temperature':0.8}
}
print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")




# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]['system']

    # 2교시에서 배운 기본 패턴
    llm = ChatOpenAI(model="gpt-4o-mini",temperature=PERSONAS[persona_name]['temperature'])

    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 {role}입니다."),
        ("human", "{question}")  
    ])

    chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})
    
    # temperature 확인용 출력
    print(chain.last.temperature)
    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content 
            







# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")
            

✅ import 완료
✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


In [8]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

None
[분석가] 안녕하세요! 데이터 분석과 관련된 질문이 있으시면 언제든지 말씀해 주세요. 도움이 되겠습니다.

✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
None
[디자이너] 안녕하세요! 어떻게 도와드릴까요? 사용자 경험에 대한 질문이 있다면 언제든지 말씀해 주세요. 😊

👋 종료합니다.
